# Tables 1, 4, 5, 6 and 7

Lexical readouts of the 14 image models and the four encoders, at the final layer and across depth.

Inputs: `scripts/analysis/table1_models.py`, which writes `results/retrieval.parquet`, `results/purity.parquet`, `results/letter_cluster.parquet`, `results/items.parquet`, `results/latent_categories.parquet`, `results/depth.parquet` and `results/encoder_lexical.parquet`; `lexicon/words.parquet` and `lexicon/pairs.parquet` from `scripts/stimuli/wordlists.py`.

In [ ]:
import numpy as np
import pandas as pd

from src import tables
from src.lexicon import edit_distance
from src.models import DEFAULT_SWEEP, ENCODERS, MODELS, PRETTY
from src.utils import setup_notebook
from src.utils.latex import BASELINE, MARK_SIGMA
from src.utils.results import pairs, read, words

setup_notebook()

retrieval = read("retrieval")
purity = read("purity")
letter_cluster = read("letter_cluster")
items = read("items")
latent_categories = read("latent_categories")
depth = read("depth")
encoder_lexical = read("encoder_lexical")
vocabulary = words()
word_pairs = pairs()

SWEEP = list(DEFAULT_SWEEP)
SELF_SUPERVISED = [m for m in SWEEP if MODELS[m].kind in ("dino", "mae")]
QWEN3 = [key for key, (_, _, kind) in ENCODERS.items() if kind == "qwen3"]
# The four models Section 4 and Appendix C name together.
LEAD = ["dinov3-h", "dinov3-7b", "clip-b", "siglip2-b"]


def named(values, form="{0:.3f} ({0:.0%})"):
    """Model names followed by their values, as in 'DINOv3-H 0.600 (60%)'."""
    return ", ".join(f"{PRETTY[m]} " + form.format(v) for m, v in values.items())

## Table 1

Neighbor purity on the noun and brand lists and forward top-10 accuracy at the final layer.

In [ ]:
table1 = tables.table1_models(purity, retrieval, encoder_lexical)
table1.write()
table1.frame

In [ ]:
# Values quoted in Section 4. Forward top-10 accuracy of the models and encoders,
# with plurals and bases pooled over their pairs.
forward = retrieval[retrieval.direction.eq("forward")]
pooled = (forward[forward.relation.isin(["negation_base", "singular_plural"])]
          .groupby("model")[["hits10", "n"]].sum())
accuracy = pd.concat([
    forward.pivot(index="model", columns="relation", values="rate10")
    .assign(plural_or_base=pooled.hits10 / pooled.n),
    encoder_lexical[encoder_lexical.readout.eq("rate10")
                    & encoder_lexical.direction.eq("forward")]
    .pivot(index="model", columns="group", values="estimate")
    .rename(columns={"negation_base+singular_plural": "plural_or_base"})])
capital, synonym = accuracy.country_capital, accuracy.synonym
one_edit, plural_or_base = accuracy.spelling_twin, accuracy.plural_or_base
pure = purity.pivot(index="model", columns="axis", values=["purity", "se", "p"])

n_nouns, n_brands = (int(purity[purity.axis.eq(axis)].n.iloc[0])
                     for axis in ("category", "brand"))
print(f"nouns {n_nouns}, brands {n_brands}; a retrieval query ranks the other "
      f"{len(vocabulary) - 1:,} words, names and pseudowords")
print("capital top-10 accuracy:", named(capital[["dinov3-h", "dinov3-7b", "clip-b",
                                                 "dinov2-b", "dinov2-l", "dinov2-g"]]))
print("neighbor purity, nouns and brands:", ", ".join(
    f"{PRETTY[m]} {pure.purity.category[m]:.3f} and {pure.purity.brand[m]:.3f}"
    for m in ["dinov3-h", "dinov3-7b", "clip-b"]))


def excess(value, se):
    """A value minus the untrained ViT-B's, in standard errors of the difference."""
    return (value - value[BASELINE]) / np.sqrt(se ** 2 + se[BASELINE] ** 2)


top10 = forward.pivot(index="model", columns="relation", values=["rate10", "rate10_se"])
four = pd.DataFrame({"Nouns": pure.purity.category, "Brands": pure.purity.brand,
                     "Capital": top10.rate10.country_capital,
                     "Synonym": top10.rate10.synonym})
against = pd.DataFrame({
    "Nouns": excess(pure.purity.category, pure.se.category),
    "Brands": excess(pure.purity.brand, pure.se.brand),
    "Capital": excess(top10.rate10.country_capital, top10.rate10_se.country_capital),
    "Synonym": excess(top10.rate10.synonym, top10.rate10_se.synonym)})
others = [m for m in SELF_SUPERVISED if m not in ("dinov3-l", "dinov3-h", "dinov3-7b")]
# Excess over the untrained ViT-B, in standard errors of the difference.
display(against.loc[["dinov3-l"] + others].rename(index=PRETTY).rename_axis("Model"))
weaker = int((four.loc["dinov3-l"] < four.loc[["dinov3-h", "dinov3-7b"]].min()).sum())
print("DINOv3-L: " + ", ".join(f"{c.lower()} {four.loc['dinov3-l', c]:.3f}" for c in four)
      + f"; below DINOv3-H and DINOv3-7B in {weaker} of {len(four.columns)} columns; "
      "excess over the untrained ViT-B in standard errors of the difference: "
      + ", ".join(f"{c.lower()} {against.loc['dinov3-l', c]:.2f}" for c in against))
print(f"the other {len(others)} self-supervised models, excess over the untrained ViT-B "
      f"on nouns, brands, capital and synonym: {against.loc[others].min().min():.2f} to "
      f"{against.loc[others].max().max():.2f} standard errors of the difference; "
      f"underlining needs more than {MARK_SIGMA}")
print("synonym top-10 accuracy: " + named(synonym[["clip-b", "dinov3-7b"] + QWEN3])
      + f"; highest of the {len(SWEEP)} models {PRETTY[synonym[SWEEP].idxmax()]}")

chance10 = float(forward.chance10.iloc[0])
print(f"one-edit word top-10 accuracy: untrained ViT-B {one_edit[BASELINE]:.3f} "
      f"({one_edit[BASELINE]:.0%}); lowest of the {len(SWEEP)} models "
      f"{one_edit[SWEEP].min():.3f}, chance {chance10:.3f}")
print("one-edit word, plural and base (plural or base pooled), DINOv2 and MAE:", ", ".join(
    f"{PRETTY[m]} {one_edit[m]:.3f}, {accuracy.singular_plural[m]:.3f} and "
    f"{accuracy.negation_base[m]:.3f} ({plural_or_base[m]:.3f})"
    for m in SWEEP if MODELS[m].family in ("DINOv2", "MAE")))


def lengths(relation):
    """The word lengths of both sides of a relation's pairs."""
    frame = word_pairs[word_pairs.relation.eq(relation)]
    return frame.a.str.len(), frame.b.str.len(), frame


anchor, twin, frame = lengths("spelling_twin")
replaced = (anchor.eq(twin) & pd.Series([edit_distance(a, b) == 1 for a, b in
                                         zip(frame.a, frame.b)], index=frame.index))
print(f"one-edit words that replace one letter and keep the anchor's length: "
      f"{int(replaced.sum())} of {len(frame)}")
print("plurals and bases that add or remove letters:", ", ".join(
    f"{name} {int(a.ne(b).sum())} of {len(a)}" for name, (a, b, _) in
    (("plurals", lengths("singular_plural")), ("bases", lengths("negation_base")))))

print(f"SigLIP2-B: plural or base {plural_or_base['siglip2-b']:.3f} "
      f"({plural_or_base['siglip2-b']:.0%}), capital {capital['siglip2-b']:.3f} "
      f"({capital['siglip2-b']:.0%}); DINOv3-H capital {capital['dinov3-h']:.3f} "
      f"({capital['dinov3-h']:.0%})")
print("one-edit word top-10 accuracy, Qwen3:", named(one_edit[QWEN3]))

## Table 4

Top-10 accuracy for every relation, with the first word of each pair as the query (forward) and with its partner (reverse).

In [ ]:
table4 = tables.table4_lexical_retrieval(retrieval, encoder_lexical)
table4.write()
table4.frame

In [ ]:
# Values behind Appendix C.
both = retrieval.pivot(index=["relation", "model"], columns="direction", values="rate10")
for relation, name, models in (("country_capital", "capital", ["dinov3-h", "dinov3-7b", "clip-b"]),
                               ("synonym", "synonym", ["dinov3-h", "dinov3-7b"])):
    print(f"{name}, forward to reverse:", ", ".join(
        f"{PRETTY[m]} {both.loc[(relation, m), 'forward']:.3f} to "
        f"{both.loc[(relation, m), 'backward']:.3f}" for m in models))

## Table 5

How often the synonym is nearer the anchor than a one-edit word, an antonym or a base form, the cosine of the anchor with its one-edit word and with its synonym in standard deviations above the vocabulary's mean cosine, and synonym top-10 hits for concrete and abstract anchors.

In [ ]:
table5 = tables.table5_items(items, retrieval, encoder_lexical)
table5.write()
table5.frame

In [ ]:
# Values behind Appendix C.
head = items[items.record_type.eq("head_to_head")]
wins = head.pivot(index="model", columns="block", values="wins").astype(int).loc[SWEEP]
holm = head.pivot(index="model", columns="block", values="sign_p_holm").loc[SWEEP]
size = head.groupby("block").n.first().astype(int)
family = head.groupby("block").model.nunique()
z = head[head.block.eq("synonym_vs_twin")].set_index("model")
contrast = items[items.record_type.eq("class_contrast")]
rest = [m for m in SWEEP if m not in LEAD]

print("models in each Holm correction: one-edit word "
      f"{family.synonym_vs_twin}, antonym {family.synonym_vs_antonym}, "
      f"base {family.synonym_vs_base}")
print(f"synonym nearer than the one-edit word: at most {wins.synonym_vs_twin.max()} "
      f"of {size.synonym_vs_twin} items ({PRETTY[wins.synonym_vs_twin.idxmax()]}), "
      f"half is {size.synonym_vs_twin / 2:g}; largest Holm-adjusted p "
      f"{holm.synonym_vs_twin.max():.1e}")
print(f"synonym similarity above the vocabulary mean, in standard deviations: at least "
      f"{z.kept_above_baseline[LEAD].min():.2f} in DINOv3-H, DINOv3-7B, CLIP-B and "
      f"SigLIP2-B; at most {z.kept_above_baseline[rest].max():.2f} in the other ten "
      f"({PRETTY[z.kept_above_baseline[rest].idxmax()]})")
print(f"synonym against antonym: smallest Holm-adjusted p "
      f"{holm.synonym_vs_antonym.min():.3f} ({PRETTY[holm.synonym_vs_antonym.idxmin()]})")
print(f"synonym nearer than the base, of {size.synonym_vs_base}:",
      named(wins.synonym_vs_base, "{0}"))
def p_value(p):
    """A p-value at three decimals, or in scientific notation below 0.001."""
    return f"{p:.3f}" if p >= 0.001 else f"{p:.1e}"


prefers_base = [m for m in SWEEP if MODELS[m].family in ("DINOv3", "CLIP", "SigLIP2",
                                                         "Web-DINO")]
no_preference = [m for m in SWEEP if MODELS[m].family in ("DINOv2", "MAE", "Untrained")]
for group, members in (("DINOv3, CLIP-B, SigLIP2-B and Web-DINO-1B", prefers_base),
                       ("DINOv2, MAE and the untrained ViT-B", no_preference)):
    print(f"{group}: {wins.synonym_vs_base[members].min()} to "
          f"{wins.synonym_vs_base[members].max()} of {size.synonym_vs_base}, half is "
          f"{size.synonym_vs_base / 2:g}; Holm-adjusted p "
          f"{p_value(holm.synonym_vs_base[members].min())} to "
          f"{p_value(holm.synonym_vs_base[members].max())}")
print(f"synonym anchors: {int(contrast.n_concrete.iloc[0])} concrete, "
      f"{int(contrast.n_abstract.iloc[0])} abstract")
# Holm-adjusted sign-test p over the 14 models, per comparison.
holm[["synonym_vs_twin", "synonym_vs_antonym", "synonym_vs_base"]].rename(
    index=PRETTY, columns={"synonym_vs_twin": "Synonym vs one-edit word",
                           "synonym_vs_antonym": "Synonym vs antonym",
                           "synonym_vs_base": "Synonym vs base"}).rename_axis(
    index="Model", columns=None)

## Table 6

Neighbor purity and held-out nearest-centroid accuracy on both lists, each with a p from a null that shuffles categories within word-length terciles.

In [ ]:
table6 = tables.table6_category(purity, latent_categories, encoder_lexical)
table6.write()
table6.frame

In [ ]:
# Values behind Section 4 and Appendix C.
held = latent_categories.pivot(
    index="model", columns="dataset",
    values=["full_classification_accuracy",
            "length_stratified_full_classification_accuracy_p"])
accuracy_held = held.full_classification_accuracy
accuracy_p = held.length_stratified_full_classification_accuracy_p
columns = pd.DataFrame({"Nouns purity": pure.purity.category,
                        "Brands purity": pure.purity.brand,
                        "Nouns accuracy": accuracy_held.common_noun,
                        "Brands accuracy": accuracy_held.brand}).loc[SWEEP]
ranks = columns.rank(ascending=False, method="min").astype(int)
display(ranks.rename(index=PRETTY).rename_axis("Model"))
print("ranks 1 to 4 in every column:",
      ", ".join(PRETTY[m] for m in ranks.index[ranks.max(axis=1).le(4)]))
print("rank 5 in every column:",
      ", ".join(PRETTY[m] for m in ranks.index[ranks.eq(5).all(axis=1)]))

n_perm = int(purity.n_perm.iloc[0])
floor = 1 / (n_perm + 1)
p = pd.DataFrame({"Nouns purity": pure.p.category, "Brands purity": pure.p.brand,
                  "Nouns accuracy": accuracy_p.common_noun,
                  "Brands accuracy": accuracy_p.brand}).loc[LEAD]
print(f"permutations per test: neighbor purity {n_perm:,}, nearest-centroid accuracy "
      f"{int(latent_categories.n_perm.iloc[0]):,}; smallest possible p "
      f"1/{n_perm + 1:,} = {floor:.4f}, reached when every permutation scores below "
      f"the observed; tests at that p for DINOv3-H, DINOv3-7B, CLIP-B and SigLIP2-B: "
      f"{int(np.isclose(p, floor).sum())} of {p.size}")
print(f"untrained ViT-B neighbor purity p: nouns {pure.p.category[BASELINE]:.3f}, "
      f"brands {pure.p.brand[BASELINE]:.3f}")

within = letter_cluster.set_index("model")
print(f"noun purity within the query's spelling cluster: chance "
      f"{within.within_cluster_chance.iloc[0]:.3f} "
      f"({within.within_cluster_chance.iloc[0]:.0%}), "
      + named(within.within_cluster[["dinov3-h", "dinov3-7b", BASELINE]]))

## Table 7

Noun neighbor purity and capital top-10 accuracy at the nine depths after the input.

In [ ]:
table7 = tables.table7_lexical_depth(depth, purity, retrieval)
table7.write()
table7.frame

In [ ]:
# Values behind Section 4 and Appendix C.
layers = depth[depth.layer_index.gt(0)]
labels = layers.groupby("layer_index").relative_depth.mean()
position = {f"{d:.2f}": i for i, d in labels.items()}
final, before = labels.index[-1], labels.index[-2]
print(f"column labels, mean relative depth over the {layers.model.nunique()} models:",
      ", ".join(position))

noun = layers.pivot(index="model", columns="layer_index", values="purity")
gain = noun - noun.loc[BASELINE]
near = [m for m in SWEEP if m not in LEAD + [BASELINE, "dinov3-l"]]
print(f"noun purity, largest difference from the untrained ViT-B at any depth in the "
      f"other {len(near)} models: {gain.loc[near].abs().max().max():.3f}; DINOv3-L "
      f"{gain.loc['dinov3-l', final]:.3f} at the final depth and at most "
      f"{gain.loc['dinov3-l'].drop(final).abs().max():.3f} before it")

rise = pd.DataFrame({"0.44 to 0.77": noun[position["0.77"]] - noun[position["0.44"]],
                     "0.77 to 1.00": noun[final] - noun[position["0.77"]]}).loc[LEAD]
display(rise.rename(index=PRETTY).rename_axis("Model"))
print("noun purity rise in DINOv3-H, DINOv3-7B, CLIP-B and SigLIP2-B:",
      "; ".join(f"{rise[span].min():.3f} to {rise[span].max():.3f} from {span}"
                for span in rise))

THREE = ["dinov3-h", "dinov3-7b", "clip-b"]
capital_depth = layers.pivot(index="model", columns="layer_index", values="retrieval10")
print(f"capital top-10 accuracy through 0.77 in DINOv3-H, DINOv3-7B and CLIP-B: at most "
      f"{capital_depth.loc[THREE, :position['0.77']].max().max():.3f}")
print(f"capital top-10 accuracy at {labels[before]:.2f}:",
      named(capital_depth.loc[THREE, before], "{0:.3f}"))
print("capital top-10 accuracy at the final depth:",
      named(capital_depth.loc[THREE, final], "{0:.3f}"))

n_capitals = int(forward[forward.relation.eq("country_capital")].n.iloc[0])
first = (layers.pivot(index="model", columns="layer_index", values="retrieval")
         * n_capitals).round().astype(int)
print(f"capitals ranked first at {labels[before]:.2f}, of {n_capitals}:",
      named(first.loc[THREE, before], "{0}"))
print("capitals ranked first before that depth, DINOv3-H, DINOv3-7B and CLIP-B:",
      first.loc[THREE, :position["0.77"]].to_numpy().sum())
# Capitals ranked first at each depth.
first.loc[THREE].rename(
    index=PRETTY, columns=lambda i: f"{labels[i]:.2f}").rename_axis(
    index="Model", columns="Relative depth")